# Working with Parallel in Pixeltable

Pixeltable's [Parallel](https://parallel.ai/) integration gives your tables and agents web search and page
extraction, as computed columns that run once per row and are cached with the rest of your data.

The UDFs work without a Parallel account: with no API key configured, they call Parallel's free MCP endpoint,
which serves searches in Fast mode under the service's anonymous rate limits. With a key, they call the Search and
Extract REST APIs, which also accept the settings shown further down.

### Prerequisites

- `pip install mcp`, which the free endpoint needs
- Optionally, a Parallel account and API key (https://platform.parallel.ai/)

### Important notes

- Parallel usage may incur costs based on your Parallel plan. Searches in `basic` and `advanced` mode cost five
  times those in `fast` and `turbo` mode.
- Be mindful of sensitive data and consider security measures when integrating with external services.

In [ ]:
%pip install -qU pixeltable mcp

Everything up to [Searching with your own account](#searching-with-your-own-account) runs without a
Parallel API key, on the free endpoint, in Fast mode. That section needs one.

In [2]:
import os

if 'PARALLEL_API_KEY' in os.environ:
    print('Using the Search and Extract APIs of your Parallel account.')
else:
    print('No PARALLEL_API_KEY: using the free endpoint, in Fast mode.')

No PARALLEL_API_KEY: using the free endpoint, in Fast mode.


Now let's create a Pixeltable directory to hold the tables for our demo.

In [3]:
import pixeltable as pxt

# Remove the 'parallel_demo' directory and its contents, if it exists
pxt.drop_dir('parallel_demo', force=True)
pxt.create_dir('parallel_demo')

Created directory 'parallel_demo'.


## Searching the web

`parallel.search()` takes the information a row needs and returns the sources that answer it, each with the
passages relevant to the request. Pass `search_queries` to rank sources against keyword queries of your own;
with none, the UDF sends the objective as the only query.

Each row also carries a conversation ID, which the next section uses.

In [4]:
import uuid
from pixeltable.functions import parallel

research_t = pxt.create_table(
    'parallel_demo.research',
    {'question': pxt.String, 'conversation': pxt.String},
)
research_t.add_computed_column(
    sources=parallel.search(
        research_t.question, session_id=research_t.conversation
    )
)

Created table 'research'.
Added 0 column values with 0 errors in 0.01 s


No rows affected.

In [5]:
research_t.insert(
    [
        {
            'question': 'What is the Pixeltable Python library for?',
            'conversation': uuid.uuid4().hex,
        },
        {
            'question': 'Which file formats can Pixeltable index?',
            'conversation': uuid.uuid4().hex,
        },
    ]
)

Inserted 2 rows with 0 errors in 1.46 s (1.37 rows/s)


2 rows inserted.

The response is a dictionary, so a JSON path reaches into it from a query or another computed column.
`results` is ordered by decreasing relevance.

In [6]:
research_t.select(
    research_t.question,
    title=research_t.sources.results[0].title,
    url=research_t.sources.results[0].url,
).collect()

question,title,url
What is the Pixeltable Python library for?,What is Pixeltable?,https://docs.pixeltable.com/
Which file formats can Pixeltable index?,Table,https://docs.pixeltable.com/sdk/latest/table


## Extracting a page

`parallel.extract()` reads one page. It takes the excerpts relevant to an objective, or the whole page as
markdown with `full_content=True`. Chaining it onto the search above extracts the top source of each row.
Parallel may read the page live; `allow_live_fetch=False` restricts the free endpoint to the copy in its
index, which answers faster but can be older.

`session_id` is the application's own ID for one conversation: a UUID, or a hex string of at least 32
characters, generated once and reused across the related searches and extractions. Parallel correlates the
calls under it, and meters the free endpoint by it. The `conversation` column holds one per row, so a row's
search and extraction go under the same ID.

In [7]:
research_t.add_computed_column(
    top_page=parallel.extract(
        research_t.sources.results[0].url,
        objective=research_t.question,
        session_id=research_t.conversation,
    )
)

Added 2 column values with 0 errors in 0.99 s (2.02 rows/s)


2 rows updated.

In [8]:
page = research_t.select(research_t.top_page).collect()['top_page'][0]
page['results'][0]['excerpts'][0][:400]

'# What is Pixeltable?\n> Pixeltable is a declarative, incremental data infrastructure for multimodal AI, unifying tables, computed columns, and vector search in one system.\n**The only open source Python library providing declarative data infrastructure for building multimodal AI applications, enabling incremental storage, transformation, indexing, retrieval, and orchestration of data.**\nWith Pixelt'

A page that could not be read is reported in `errors` rather than failing the row, so one dead link does
not stop the rest of a batch.

In [9]:
research_t.select(
    url=research_t.top_page.results[0].url,
    errors=research_t.top_page.errors,
).collect()

url,errors
https://docs.pixeltable.com/,[]
https://docs.pixeltable.com/sdk/latest/table,[]


## Searching with your own account

With `PARALLEL_API_KEY` set, `search()` also takes `mode`, `max_results`, `source_policy` and `fetch_policy`,
and `extract()` takes `fetch_policy` and `links`. Requesting one of them without a key reports
`MISSING_CREDENTIALS` instead of silently searching on the free tier, which is why the cell below runs only
with a key.

`mode` defaults to `fast`. Use `advanced` for multi-hop questions, `turbo` for the lowest latency, and read
<https://docs.parallel.ai/search/modes> for the trade-offs.

In [10]:
if 'PARALLEL_API_KEY' in os.environ:
    research_t.add_computed_column(
        docs_only=parallel.search(
            research_t.question,
            mode='advanced',
            max_results=3,
            source_policy={'include_domains': ['docs.pixeltable.com']},
        )
    )
    display(
        research_t.select(
            url=research_t.docs_only.results[0].url
        ).collect()
    )
else:
    print('These settings need PARALLEL_API_KEY; skipping.')

These settings need PARALLEL_API_KEY; skipping.


## Giving a model web access

Both UDFs go into a tool list like any other Pixeltable function, and a model calls them through the usual
[tool-calling API](/howto/cookbooks/agents/llm-tool-calling):

```python
tools = pxt.tools(parallel.search, parallel.extract)
```

A published tool exposes every parameter, so the model chooses the search settings and invents a `session_id`
of its own. Binding a parameter fixes its value and takes it out of the published schema, which is how an
application keeps the conversation ID, and the settings its credentials cannot serve, to itself:

```python
conversation = uuid.uuid4().hex
tools = pxt.tools(
    pxt.tool(
        parallel.search.using(
            mode='fast',
            max_results=None,
            source_policy=None,
            fetch_policy=None,
            session_id=conversation,
        ),
        name='search',
    ),
    pxt.tool(
        parallel.extract.using(
            fetch_policy=None,
            links=None,
            allow_live_fetch=None,
            session_id=conversation,
        ),
        name='extract',
    ),
)
```

That leaves the model `objective` and `search_queries` on `search`, and `url`, `objective`, `search_queries`
and `full_content` on `extract`: everything the free endpoint serves, and nothing it refuses.

[Agentic workflows](/use-cases/agentic-workflows) puts the whole loop together.

## Learn More

- [Parallel documentation](https://docs.parallel.ai/)
- [Search API](https://docs.parallel.ai/search/search-quickstart)
- [Extract API](https://docs.parallel.ai/extract/extract-quickstart)
- [Search modes](https://docs.parallel.ai/search/modes)
- [Parallel MCP server](https://docs.parallel.ai/integrations/mcp/quickstart)